Imports

In [1]:
%matplotlib inline
import pandas as pd
import numpy as np
import kagglehub
from umap import UMAP
from kagglehub import KaggleDatasetAdapter
from sentence_transformers import SentenceTransformer
from hdbscan import HDBSCAN
from bertopic import BERTopic
from bertopic.vectorizers import OnlineCountVectorizer
from langchain_google_genai import ChatGoogleGenerativeAI
from bertopic.representation import LangChain

C:\Users\gianf\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Load Data

In [2]:
df = pd.read_csv('Data/combined_timeline_data.csv')
df.head()

,Date,Source,Summary,Associated Link Title,Associated Link URL,Theme
0,"January 4, 2020",WHO Announces Pneumonia Cases of Unknown Cause,The World Health Organization (WHO) announces ...,WHO on Twitter,https://fraser.stlouisfed.org/docs/historical/...,Pandemic
1,"January 8, 2020",CDC Issues Health Advisory,The Centers for Disease Control and Prevention...,CDC Health Advisory,https://fraser.stlouisfed.org/archival-collect...,Pandemic
2,"January 9, 2020",CDC Notes Appearance of Novel Coronavirus Outb...,The CDC notes the appearance of a novel corona...,CDC Report,https://fraser.stlouisfed.org/archival-collect...,Pandemic
3,"January 17, 2020, 2:00 pm EST",CDC Announces Enhanced Screenings for Those Tr...,The CDC and the Department of Homeland Securit...,CDC Press Release,https://fraser.stlouisfed.org/archival-collect...,Pandemic
4,"January 21, 2020",Washington State Department of Health Announce...,The Washington State Department of Health anno...,Snohomish Health District Media Release,https://fraser.stlouisfed.org/title/state-mate...,Pandemic


In [8]:
docs = df["Summary"].tolist()
# Extract only the date part (remove everything after the year)
timestamps = pd.to_datetime(df["Date"].str.extract(r'([A-Za-z]+\s+\d{1,2},\s+\d{4})')[0]).dt.date

Prepare Models

In [9]:
sentence_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = sentence_model.encode(docs, show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 968.75it/s, Materializing param=pooler.dense.weight]                              
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 968.75it/s, Materializing param=pooler.dense.weight]                
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 17/17 [00:02<00:00,  6.12it/s]


Train BERTopic

In [6]:
print("Training BERTopic model...\n")


umap_model = UMAP(n_neighbors=15, n_components=10, metric='cosine')
hdbscan_model = HDBSCAN(min_cluster_size=10, metric='euclidean', cluster_selection_method='eom', prediction_data=True)
vectorizer_model = OnlineCountVectorizer(stop_words="english")


topic_model = BERTopic(
  embedding_model=sentence_model,
  umap_model=umap_model,
  hdbscan_model=hdbscan_model,
  vectorizer_model=vectorizer_model,
)


# Train BERTopic
topic_model = topic_model.fit(docs, embeddings)


print("\nModel training complete!")

Training BERTopic model...


Model training complete!


Get Topic Information

In [7]:
# Get topic information
topic_model.get_topic_info()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,121,-1_bank_billion_reserve_federal,"[bank, billion, reserve, federal, new, aig, sw...",[The Federal Reserve Board and the U.S. Treasu...
1,0,78,0_covid_19_cdc_states,"[covid, 19, cdc, states, united, reports, peop...","[The CDC reports that over 500,000 people have..."
2,1,65,1_financial_federal_reserve_capital,"[financial, federal, reserve, capital, board, ...",[The Federal Reserve Board provides additional...
3,2,49,2_facility_backed_securities_talf,"[facility, backed, securities, talf, asset, ma...","[The Commercial Paper Funding Facility, Asset-..."
4,3,28,3_rate_percent_votes_points,"[rate, percent, votes, points, target, basis, ...",[The FOMC votes to reduce its target for the f...
5,4,28,4_commission_developments_report_recent,"[commission, developments, report, recent, con...",[The Congressional Oversight Commission releas...
6,5,26,5_tarp_troubled_relief_asset,"[tarp, troubled, relief, asset, program, treas...",[The Office of the Special Inspector General f...
7,6,22,6_mae_fannie_freddie_mac,"[mae, fannie, freddie, mac, billion, loss, net...",[Fannie Mae reports a loss of $25.2 billion in...
8,7,22,7_purchases_stock_total_preferred,"[purchases, stock, total, preferred, purchase,...",[The U.S. Treasury Department purchases a tota...
9,8,20,8_ppp_small_loans_paycheck,"[ppp, small, loans, paycheck, program, protect...",[The Federal Reserve announces a second extens...


Visualize Topics

In [8]:
topic_model.visualize_documents(docs, embeddings=embeddings)

Visualize Topic Similarity

In [9]:
fig = topic_model.visualize_heatmap()
fig.show()

Hierarchichal Topic Visualization

In [10]:
hierarchical_topics = topic_model.hierarchical_topics(docs)
fig = topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)
fig.show()

print("💡 This shows the hierarchical structure of topics")

100%|██████████| 12/12 [00:00<00:00, 220.86it/s]


💡 This shows the hierarchical structure of topics


Visualize Topics over Time

In [11]:
topics_over_time = topic_model.topics_over_time(docs, timestamps)
topic_model.visualize_topics_over_time(topics_over_time)

2026-01-27 17:07:27,932 - BERTopic - WARNING: There are more than 100 unique timestamps (i.e., 408) which significantly slows down the application. Consider setting `nr_bins` to a value lower than 100 to speed up calculation. 
